# Change Data Capture demo
Shows how Delta CDF captures inserts, updates and deletes on a small demo table

### Setup

In [0]:
dbutils.widgets.text("delta_catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("delta_schema", "oles0305_lab10")

In [0]:
delta_catalog = dbutils.widgets.get("delta_catalog")
delta_schema = dbutils.widgets.get("delta_schema")

demo_table = f"{delta_catalog}.{delta_schema}.customer_demo"

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {demo_table}
    AS
    SELECT * 
    FROM {delta_catalog}.{delta_schema}.customer_snapshot
    ORDER BY CustomerID
    LIMIT 10
""")

### Enable CDF

In [0]:
spark.sql(f"""
    ALTER TABLE {demo_table}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

### State before changes

In [0]:
def get_latest_version(table):
    return spark.sql(f"""
        DESCRIBE HISTORY {table}
    """).orderBy("version", ascending=False).limit(1).collect()[0].version

baseline_version = get_latest_version(f"{demo_table}")

In [0]:
spark.sql(f"DESCRIBE HISTORY {demo_table}") \
    .select("version", "timestamp", "operation") \
    .display()

In [0]:
spark.sql(f"""
    SELECT CustomerID, FirstName, LastName
    FROM {demo_table}
    WHERE CustomerID BETWEEN 1 AND 5
""").display()

### Simulate changes

In [0]:
spark.sql(f"""
    UPDATE {demo_table} 
    SET FirstName = 'UpdatedName' WHERE CustomerID = 1
""")

spark.sql(f"""
    DELETE FROM {demo_table}
    WHERE CustomerID = 2
""")

spark.sql(f"""
    INSERT INTO {demo_table} (CustomerID, FirstName, LastName)
    VALUES (99999, 'NewName', 'NewLast')
""")

### State after changes and table history

In [0]:
spark.sql(f"""
    SELECT CustomerID, FirstName, LastName
    FROM {demo_table}
    WHERE CustomerID IN (1, 2, 3, 4, 5, 99999)
    ORDER BY CustomerID
""").display()

In [0]:
spark.sql(f"DESCRIBE HISTORY {demo_table}") \
    .select("version", "timestamp", "operation") \
    .display()

### Captured change events

In [0]:
changes = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", baseline_version + 1)
        .table(demo_table)
)

changes.select("CustomerID", "FirstName", "LastName", 
               "_change_type", "_commit_version", "_commit_timestamp") \
    .orderBy("_commit_version", "CustomerID") \
    .display()